In [1]:
%run src/donnees.py

# df_caract_recoder
# df_lieux_recoder
# df_vehicule_recoder
# df_usager_recoder
# df_final

Problématique : Quels facteurs influencent la gravité des accidents corporels de la route en France, peut-on prédire cette gravité en tenant compte des caractéristiques de l'accident (environnementales, géographique...) ?

# Cartographie

Nous souhaitons voir si certaines régions (ou departements) ont plus d'accidents grave ... ?
Par accident, il peut y avoir plusiuers victime, nous considérons donc le nombre de victime et non le nombre d'accident 

In [2]:
# nombre d'accident
df_final["Num_Acc"].nunique()

NameError: name 'df_final' is not defined

In [ ]:
# nombre de victime 
df_final["id_usager"].nunique()

In [ ]:
# recupération du nombre de victime par dep et grav  (utilisation longitute lattiude ?)
df_victime_dep = df_final.groupby(["dep", "grav"]).size().reset_index(nComme oComme cccccomeme Com on Comme on peut dllnnnnnpekfef fffe5lllllame="nb")

In [ ]:
df_victime_dep

In [ ]:
df_final["grav"].unique()
# suppression des nan - automatique avec le groupby fait

In [ ]:
df_victime_tue = df_victime_dep[df_victime_dep["grav"] == "Tué"]

In [ ]:
df_victime_tue

Carte des accidents grave puis carte avec chaque type d'accident
toute année confondu 

In [ ]:
import matplotlib.pyplot as plt
!pip install cartiflette
from cartiflette import carti_download

def initialisation_carte():
    departement_borders = carti_download(
        values=["France"],
        crs=4326,
        borders="DEPARTEMENT",
        vectorfile_format="geojson",
        simplification=50,
        filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
        source="EXPRESS-COG-CARTO-TERRITOIRE",
        year=2022)

    departement_borders = departement_borders.rename(columns={"INSEE_DEP": "dep"})
    return departement_borders




def carte(blessure, df):
    """
    Fonction pour créer une carte par blessure
    """
    if blessure == "non":
        df_blessure = df.groupby(["dep"]).size().reset_index(name="nb")
    else:
        df_blessure = df[df["grav"] == blessure]

    carte = initialisation_carte().merge(
        df_blessure,
        on="dep",
        how="left"
    )

    # Carte
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    carte.plot(
            column="nb",
            cmap="RdBu_r",
            legend=True,
            edgecolor="black",
            linewidth=0.5,
            ax=ax
        )

    ax.set_title(f"Nombre de {blessure} par département sur toute les années", fontsize=16)
    ax.axis("off")

    plt.show()

    

Une carte sur le nombre d'accident pour voir la répartition 
Une carte toute victime confondu ?

In [ ]:
carte("Tué",df_victime_dep)

In [ ]:
carte("Blessé hospitalisé",df_victime_dep)

In [ ]:
carte("Indemne",df_victime_dep)

In [ ]:
carte("Blessé léger",df_victime_dep)

In [ ]:
carte("non",df_final)

Il y a bcp d'accident en ile de france, interressantde plutot faire un pourcentage des accidnets par region 

In [ ]:
df_victime_tot = df_final.groupby(["dep"]).size().reset_index(name="nb").sort_values(by="nb")

In [ ]:
df_victime_tot

Proportion d'une catégorie de blessure dans le nb de victime total par dep
(faire nb d'accidnet ? en mode un mortel = au moins une victime ?)

In [ ]:
# enlever les na ????
df_victimes_total = (
    df_final
    #.dropna(subset=["dep", "grav"])
    .groupby("dep")
    .size()
    .reset_index(name="nb_victimes")
)

df_victime_dep2 = df_final.groupby(["dep", "grav"]).size().reset_index(name="nb")

df_dep = df_victime_dep2.merge(df_victimes_total, on="dep", how="left")
df_dep["proportion"] = df_dep["nb"] / df_dep["nb_victimes"] *100

In [ ]:
df_dep[df_dep["grav"] == "Tué"].sort_values(by="proportion")

In [ ]:
df_victimes_total

In [ ]:
import matplotlib.ticker as mtick
def carte2(blessure, df):
    """
    Fonction pour créer une carte par blessure proportion par dep
    """

    df_blessure = df[df["grav"] == blessure]

    carte = initialisation_carte().merge(
        df_blessure,
        on="dep",
        how="left"
    )

    # Carte
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    m=carte.plot(
            column="proportion",
            cmap="RdBu_r",
            legend=True,
            edgecolor="black",
            linewidth=0.5,
            ax=ax
        )

        # Récupérer la colorbar
    cbar = m.get_figure().axes[-1]

    # Formater en pourcentage
    cbar.yaxis.set_major_formatter(mtick.PercentFormatter())


    ax.set_title(f"Proportion de {blessure} par département sur toute les années", fontsize=16)
    ax.axis("off")

    plt.show()


    

In [ ]:
carte2("Tué", df_dep)

In [ ]:
df_dep[df_dep["grav"] == "Tué"].sort_values(by="proportion")

In [ ]:
df_dep_an[df_dep_an["grav"] == "Tué"].sort_values(by="proportion")

In [ ]:
df_dep.sort_values(by="proportion")

In [ ]:
carte2("Indemne", df_dep)

In [ ]:
carte2("Tué",df_dep_an[df_dep_an["an"]==2023])

Les victimes tués par un accdient sont une faible proportion des victimes, ou on en voit le plus dans la diagonal du vide 
En revanche, les accidents indemne sont plus vers le nord 
comparaison année à faire 

In [ ]:
df_victimes_total_an = (
    df_final
    .groupby(["an", "dep"])
    .size()
    .reset_index(name="nb_victimes")
)

df_victime_dep2_an = df_final.groupby(["an", "dep", "grav"]).size().reset_index(name="nb")

df_dep_an = df_victime_dep2_an.merge(df_victimes_total_an, on=["an", "dep"], how="left")
df_dep_an["proportion"] = df_dep_an["nb"] / df_dep_an["nb_victimes"] *100

In [ ]:
df_victimes_total_an["an"].unique()

In [ ]:
df_victime_dep2_an["an"].unique()

In [ ]:
df_dep_an[df_dep_an["dep"]=="01"]

In [ ]:
import plotly.express as px
def carte_interactive(blessure, df):
    """
    Fonction pour créer une carte par blessure proportion par dep interactive par annéé
    """

    df_blessure = df[df["grav"] == blessure]

    # Chargement de la carte
    geo = initialisation_carte()
    max_par_an = df_blessure.groupby("an")["proportion"].max().to_dict()

    # Carte interactive
    fig = px.choropleth(
        df_blessure,
        geojson=geo,
        locations="dep",
        color="proportion",
        animation_frame="an",
        featureidkey="properties.dep",
        color_continuous_scale="RdBu_r",
        range_color=(0, df_blessure["proportion"].max()),
        labels={"proportion": "Proportion (%)"}
    )
    # 3. Appliquer un range_color différent pour chaque frame
    for frame in fig.frames:
        an = frame.name
        max_val = max_par_an[int(an)]
        frame.data[0].zmax = max_val
        frame.data[0].zmin = 0

    # 4. Appliquer aussi au layout initial
    an0 = df_blessure["an"].min()
    fig.data[0].zmax = max_par_an[an0]
    fig.data[0].zmin = 0
        # Afficher les contours des départements
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        showcountries=True,
        showsubunits=True,
        subunitcolor="black",
        subunitwidth=0.5
    )

    fig.update_layout(
        title=f"Évolution de la proportion de {blessure} par département",
        coloraxis_colorbar=dict(
            title="%",
            ticksuffix="%",
        )
    )

    fig.show()



In [ ]:
carte_interactive("Tué",df_dep_an)

In [ ]:
import plotly.express as px
import json

def carte_interactive(blessure, df):
    df_blessure = df[df["grav"] == blessure].copy()

    # Charger la carte (GeoDataFrame)
    gdf = initialisation_carte()

    # Convertir en vrai GeoJSON
    geo = json.loads(gdf.to_json())

    # Renommer la clé dans le GeoJSON
    for f in geo["features"]:
        f["properties"]["dep"] = f["properties"].pop("dep")  # car tu l'as déjà renommé dans initialisation_carte

    # Max par année
    max_par_an = df_blessure.groupby("an")["proportion"].max().to_dict()

    # Carte interactive
    fig = px.choropleth(
        df_blessure,
        geojson=geo,
        locations="dep",
        color="proportion",
        animation_frame="an",
        featureidkey="properties.dep",
        color_continuous_scale="RdBu_r",
        labels={"proportion": "Proportion (%)"}
    )

    # Échelle dynamique
    for frame in fig.frames:
        an = int(frame.name)
        frame.data[0].zmin = 0
        frame.data[0].zmax = max_par_an[an]

    an0 = df_blessure["an"].min()
    fig.data[0].zmin = 0
    fig.data[0].zmax = max_par_an[an0]

    # Contours
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        showsubunits=True,
        subunitcolor="black",
        subunitwidth=0.5
    )

    fig.update_layout(
        title=f"Évolution de la proportion de {blessure} par département",
        coloraxis_colorbar=dict(
            title="%",
            ticksuffix="%",
        )
    )

    fig.show()


In [ ]:
carte_interactive("Tué",df_dep_an)

In [ ]:
def debug_geo(gdf):
    print("TYPE :", type(gdf))
    print("COLONNES :", gdf.columns)
    print("GEOMETRY TYPE :", gdf.geometry.geom_type.unique())
    print("NB LIGNES :", len(gdf))
    try:
        geo = json.loads(gdf.to_json())
        print("GEOJSON KEYS :", geo.keys())
    except Exception as e:
        print("ERREUR to_json :", e)
debug_geo(initialisation_carte())

In [ ]:
import plotly.express as px
import json

def carte_interactive(blessure, df):
    df_blessure = df[df["grav"] == blessure].copy()

    # Harmoniser les codes départements
    df_blessure['dep'] = df_blessure['dep'].astype(str).str.zfill(2)

    # Charger la carte
    gdf = initialisation_carte()
    gdf['dep'] = gdf['dep'].astype(str).str.zfill(2)

    # Convertir en GeoJSON
    geo = json.loads(gdf.to_json())

    # Max par année
    max_par_an = df_blessure.groupby("an")["proportion"].max().to_dict()

    fig = px.choropleth(
        df_blessure,
        geojson=geo,
        locations="dep",
        color="proportion",
        animation_frame="an",
        featureidkey="properties.dep",
        color_continuous_scale="RdBu_r",
        labels={"proportion": "Proportion (%)"}
    )

    # Échelle dynamique
    for frame in fig.frames:
        an = int(frame.name)
        frame.data[0].zmin = 0
        frame.data[0].zmax = max_par_an[an]

    an0 = df_blessure["an"].min()
    fig.data[0].zmin = 0
    fig.data[0].zmax = max_par_an[an0]

    # Contours
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        showsubunits=True,
        subunitcolor="black",
        subunitwidth=0.5
    )

    fig.update_layout(
        title=f"Évolution de la proportion de {blessure} par département",
        coloraxis_colorbar=dict(
            title="%",
            ticksuffix="%",
        )
    )

    fig.show()


In [ ]:
carte_interactive("Tué",df_dep_an)